# Building a Decoder-Only Transformer from Scratch (100M Parameters)

Welcome! In this notebook, you'll learn how to build a decoder-only transformer model (like GPT) completely from scratch. We'll implement every single component and explain it with real examples.

## What is a Decoder-Only Transformer?

Think of it like this:
- **Input**: "The cat sat on the" 
- **Output**: Predicts the next word, like "mat"

A decoder-only model:
1. Takes a sequence of text as input
2. Processes it through multiple layers
3. Predicts what comes next

It's called "decoder-only" because it only has the decoder part of the original transformer (no encoder). Models like GPT-2, GPT-3, and GPT-4 are all decoder-only transformers.

## Let's Get Started!

In [ ]:
# First, let's import all the libraries we need
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import numpy as np
from torch.utils.data import Dataset, DataLoader

# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

---
## Step 1: Understanding Tokenization

**What is tokenization?**

Computers don't understand text directly. We need to convert text into numbers. Tokenization breaks text into small pieces (tokens) and assigns each a unique number.

**Example:**
- Text: "Hello world"
- Tokens: ["Hello", " world"] (note the space)
- Token IDs: [15496, 995] (these are just example numbers)

For simplicity, we'll build a character-level tokenizer (each character becomes a token).

In [ ]:
class CharacterTokenizer:
    """
    A simple character-level tokenizer.
    Each unique character gets its own ID number.
    """
    def __init__(self, text):
        # Find all unique characters in the text
        self.chars = sorted(list(set(text)))
        self.vocab_size = len(self.chars)
        
        # Create mappings: character -> number and number -> character
        self.char_to_idx = {ch: i for i, ch in enumerate(self.chars)}
        self.idx_to_char = {i: ch for i, ch in enumerate(self.chars)}
        
        print(f"Vocabulary size: {self.vocab_size}")
        print(f"Characters: {''.join(self.chars[:50])}...")
    
    def encode(self, text):
        """Convert text to a list of token IDs"""
        return [self.char_to_idx[ch] for ch in text]
    
    def decode(self, indices):
        """Convert token IDs back to text"""
        return ''.join([self.idx_to_char[i] for i in indices])

# Let's test it with an example!
sample_text = "Hello! This is a transformer. It learns patterns in text."
tokenizer = CharacterTokenizer(sample_text)

# Encode the text
encoded = tokenizer.encode("Hello")
print(f"\nOriginal text: 'Hello'")
print(f"Encoded (token IDs): {encoded}")

# Decode it back
decoded = tokenizer.decode(encoded)
print(f"Decoded back: '{decoded}'")

---
## Step 2: Token Embeddings

**What are embeddings?**

Token IDs are just numbers (like 5, 10, 15). But we need richer representations. An embedding converts each token ID into a vector of numbers that captures meaning.

**Example:**
- Token ID: 5 (maybe the character 'a')
- Embedding: [0.2, -0.5, 0.8, 0.1, ...] (a vector of size `embedding_dim`)

Think of it as giving each token a personality with multiple attributes!

**Why vectors?**
- Similar tokens will have similar vectors
- The model learns these during training

In [ ]:
class TokenEmbedding(nn.Module):
    """
    Converts token IDs into dense vectors.
    
    Parameters:
    - vocab_size: How many unique tokens we have
    - embedding_dim: Size of each embedding vector
    """
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        # This is a lookup table: for each token ID, store a vector
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.embedding_dim = embedding_dim
    
    def forward(self, x):
        """
        x: tensor of token IDs, shape (batch_size, sequence_length)
        returns: embeddings, shape (batch_size, sequence_length, embedding_dim)
        """
        # Scale embeddings by sqrt(embedding_dim) - this is a standard practice
        # It helps with training stability
        return self.embedding(x) * math.sqrt(self.embedding_dim)

# Example: Let's create embeddings for a small vocabulary
vocab_size = 50  # Let's say we have 50 unique characters
embedding_dim = 128  # Each token becomes a vector of 128 numbers

token_emb = TokenEmbedding(vocab_size, embedding_dim)

# Example input: token IDs for a sequence
# Let's say we have: "Hello" -> [15, 20, 25, 25, 30]
example_tokens = torch.tensor([[15, 20, 25, 25, 30]])  # shape: (1, 5)
embeddings = token_emb(example_tokens)

print(f"Input shape (batch_size, sequence_length): {example_tokens.shape}")
print(f"Output shape (batch_size, sequence_length, embedding_dim): {embeddings.shape}")
print(f"\nFirst token embedding (first 10 values): {embeddings[0, 0, :10]}")

---
## Step 3: Positional Embeddings

**Why do we need position information?**

Consider these two sentences:
1. "Dog bites man" 
2. "Man bites dog"

Same words, completely different meanings! The **position** of words matters.

**How it works:**
- Position 0 (first token) gets one vector: [0.1, 0.2, 0.3, ...]
- Position 1 (second token) gets another vector: [0.4, 0.1, 0.8, ...]
- And so on...

We **add** positional embeddings to token embeddings:
- Final = Token Embedding + Positional Embedding

In [ ]:
class PositionalEmbedding(nn.Module):
    """
    Adds position information to embeddings.
    We use learnable positional embeddings (the model learns the best position vectors).
    
    Parameters:
    - max_seq_len: Maximum sequence length we'll support
    - embedding_dim: Size of embedding (must match token embedding size)
    """
    def __init__(self, max_seq_len, embedding_dim):
        super().__init__()
        # Create a learnable embedding for each position
        self.pos_embedding = nn.Embedding(max_seq_len, embedding_dim)
    
    def forward(self, x):
        """
        x: embeddings from token embedding, shape (batch_size, seq_len, embedding_dim)
        returns: embeddings with position info added
        """
        batch_size, seq_len, embedding_dim = x.shape
        
        # Create position indices: [0, 1, 2, 3, ..., seq_len-1]
        positions = torch.arange(0, seq_len, device=x.device).unsqueeze(0)
        # Shape: (1, seq_len)
        
        # Get positional embeddings
        pos_emb = self.pos_embedding(positions)  # Shape: (1, seq_len, embedding_dim)
        
        # Add to input embeddings
        return x + pos_emb

# Example:
max_seq_len = 512  # Support sequences up to 512 tokens
pos_emb = PositionalEmbedding(max_seq_len, embedding_dim)

# Using our previous token embeddings
embeddings_with_pos = pos_emb(embeddings)

print(f"Before adding position: {embeddings.shape}")
print(f"After adding position: {embeddings_with_pos.shape}")
print(f"\nThe shape stays the same, but now each position has unique information!")

---
## Step 4: Understanding Attention Mechanism

**What is attention?**

Attention lets the model focus on relevant parts of the input. Think of it like this:

**Example sentence:** "The cat, which was very fluffy, sat on the mat."

When predicting what comes after "sat", the model should pay attention to:
- **"cat"** (high attention) - who is sitting?
- **"sat"** (high attention) - what action?
- **"fluffy"** (low attention) - less relevant for what comes next

**The Three Components:**
1. **Query (Q)**: "What am I looking for?" 
2. **Key (K)**: "What do I contain?"
3. **Value (V)**: "What information do I carry?"

**Intuition:**
- Each position creates a Query: "I want to find related words"
- Each position creates a Key: "Here's what I am"
- We compare Query with Keys to find matches (using dot product)
- Use those matches to weight the Values

In [ ]:
class SelfAttention(nn.Module):
    """
    Self-attention mechanism.
    Each position attends to all previous positions (and itself).
    
    Parameters:
    - embedding_dim: Size of input embeddings
    - head_dim: Size of each attention head
    """
    def __init__(self, embedding_dim, head_dim):
        super().__init__()
        self.head_dim = head_dim
        
        # Linear layers to create Q, K, V from input
        self.query = nn.Linear(embedding_dim, head_dim)
        self.key = nn.Linear(embedding_dim, head_dim)
        self.value = nn.Linear(embedding_dim, head_dim)
    
    def forward(self, x, mask=None):
        """
        x: input embeddings, shape (batch_size, seq_len, embedding_dim)
        mask: attention mask (for causal/decoder attention)
        """
        batch_size, seq_len, embedding_dim = x.shape
        
        # Step 1: Create Q, K, V
        Q = self.query(x)  # (batch_size, seq_len, head_dim)
        K = self.key(x)    # (batch_size, seq_len, head_dim)
        V = self.value(x)  # (batch_size, seq_len, head_dim)
        
        # Step 2: Compute attention scores
        # Q @ K^T gives us similarity between all pairs of positions
        scores = torch.matmul(Q, K.transpose(-2, -1))  # (batch_size, seq_len, seq_len)
        
        # Scale by sqrt(head_dim) to prevent scores from getting too large
        scores = scores / math.sqrt(self.head_dim)
        
        # Step 3: Apply mask (for decoder - can't attend to future positions)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # Step 4: Apply softmax to get attention weights
        # This converts scores to probabilities (they sum to 1)
        attention_weights = F.softmax(scores, dim=-1)  # (batch_size, seq_len, seq_len)
        
        # Step 5: Apply attention weights to values
        output = torch.matmul(attention_weights, V)  # (batch_size, seq_len, head_dim)
        
        return output, attention_weights

# Example:
head_dim = 64
attention = SelfAttention(embedding_dim, head_dim)

# Create a causal mask (for decoder - can't look into the future)
seq_len = 5
causal_mask = torch.tril(torch.ones(seq_len, seq_len)).unsqueeze(0)
print("Causal Mask (1 = can attend, 0 = cannot attend):")
print(causal_mask[0])
print("\nNotice: Each position can only attend to itself and previous positions!")

# Apply attention
output, attn_weights = attention(embeddings_with_pos[:, :5, :], mask=causal_mask)
print(f"\nAttention output shape: {output.shape}")
print(f"Attention weights shape: {attn_weights.shape}")

---
## Step 5: Multi-Head Attention

**Why multiple heads?**

Think of it like having multiple perspectives:
- **Head 1** might focus on grammar (subject-verb agreement)
- **Head 2** might focus on semantics (meaning relationships)
- **Head 3** might focus on syntax (sentence structure)

**Example:**
Sentence: "The bank is near the river"
- One head might connect "bank" with "river" (financial institution? no, riverbank!)
- Another head might connect "bank" with "near" (location relationship)

**How it works:**
1. Run multiple attention heads in parallel
2. Concatenate their outputs
3. Project back to original dimension

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-head attention: Multiple attention mechanisms running in parallel.
    
    Parameters:
    - embedding_dim: Size of input embeddings
    - num_heads: Number of attention heads
    - dropout: Dropout probability for regularization
    """
    def __init__(self, embedding_dim, num_heads, dropout=0.1):
        super().__init__()
        assert embedding_dim % num_heads == 0, "embedding_dim must be divisible by num_heads"
        
        self.embedding_dim = embedding_dim
        self.num_heads = num_heads
        self.head_dim = embedding_dim // num_heads  # Each head processes a slice
        
        # Single linear layers for all heads (more efficient than separate layers)
        self.qkv = nn.Linear(embedding_dim, 3 * embedding_dim)
        self.output_projection = nn.Linear(embedding_dim, embedding_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, mask=None):
        """
        x: input, shape (batch_size, seq_len, embedding_dim)
        """
        batch_size, seq_len, embedding_dim = x.shape
        
        # Step 1: Generate Q, K, V for all heads at once
        qkv = self.qkv(x)  # (batch_size, seq_len, 3 * embedding_dim)
        
        # Reshape and split into Q, K, V
        qkv = qkv.reshape(batch_size, seq_len, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, batch_size, num_heads, seq_len, head_dim)
        Q, K, V = qkv[0], qkv[1], qkv[2]
        
        # Step 2: Compute attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        # Shape: (batch_size, num_heads, seq_len, seq_len)
        
        # Step 3: Apply mask
        if mask is not None:
            scores = scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))
        
        # Step 4: Softmax
        attention_weights = F.softmax(scores, dim=-1)
        attention_weights = self.dropout(attention_weights)
        
        # Step 5: Apply attention to values
        output = torch.matmul(attention_weights, V)
        # Shape: (batch_size, num_heads, seq_len, head_dim)
        
        # Step 6: Concatenate heads
        output = output.transpose(1, 2).contiguous()
        output = output.reshape(batch_size, seq_len, embedding_dim)
        
        # Step 7: Final linear projection
        output = self.output_projection(output)
        
        return output

# Example:
num_heads = 8  # Common choice: 8 or 16 heads
mha = MultiHeadAttention(embedding_dim, num_heads)

output = mha(embeddings_with_pos[:, :5, :], mask=causal_mask)
print(f"Multi-head attention output shape: {output.shape}")
print(f"Number of heads: {num_heads}")
print(f"Each head dimension: {mha.head_dim}")

---
## Step 6: Feed-Forward Network

**What does it do?**

After attention, we need to process the information further. The feed-forward network is like a "thinking" step:

1. **Expand**: Project to a larger dimension (typically 4x larger)
2. **Non-linearity**: Apply activation function (GELU) to capture complex patterns
3. **Compress**: Project back to original dimension

**Why expand then compress?**
- The expansion gives the model more "space" to compute complex transformations
- It's like expanding your thinking space before making a decision

**Example intuition:**
- Input: Information about "bank" after attention
- FFN processes: Is it financial? Is it geographical? What's the context?
- Output: Refined representation of "bank" in context

In [ ]:
class FeedForward(nn.Module):
    """
    Position-wise Feed-Forward Network.
    Applied to each position independently.
    
    Parameters:
    - embedding_dim: Input/output dimension
    - ff_dim: Hidden dimension (typically 4 * embedding_dim)
    - dropout: Dropout probability
    """
    def __init__(self, embedding_dim, ff_dim, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(embedding_dim, ff_dim)
        self.linear2 = nn.Linear(ff_dim, embedding_dim)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU()  # GELU is commonly used in transformers
    
    def forward(self, x):
        """
        x: input, shape (batch_size, seq_len, embedding_dim)
        """
        # Step 1: Expand
        x = self.linear1(x)  # (batch_size, seq_len, ff_dim)
        
        # Step 2: Non-linearity
        x = self.activation(x)
        x = self.dropout(x)
        
        # Step 3: Compress back
        x = self.linear2(x)  # (batch_size, seq_len, embedding_dim)
        x = self.dropout(x)
        
        return x

# Example:
ff_dim = 4 * embedding_dim  # Standard: 4x the embedding dimension
ffn = FeedForward(embedding_dim, ff_dim)

ff_output = ffn(output)
print(f"Input shape: {output.shape}")
print(f"Output shape: {ff_output.shape}")
print(f"\nHidden dimension: {ff_dim} (4x the embedding dimension)")

# Calculate parameters
params = sum(p.numel() for p in ffn.parameters())
print(f"Parameters in this FFN: {params:,}")

---
## Step 7: Layer Normalization

**What is normalization?**

During training, values can become very large or very small, making learning unstable. Layer normalization fixes this.

**How it works:**
1. Calculate mean and standard deviation across the embedding dimension
2. Normalize: (x - mean) / std
3. Scale and shift with learnable parameters

**Example:**
- Before: [100, 200, 150, 180]
- After normalization: [~0, ~1, ~0.5, ~0.8] (approximately)
- This keeps values in a reasonable range!

**Why is it important?**
- Stabilizes training
- Allows faster learning
- Prevents gradient problems

In [ ]:
# PyTorch already provides LayerNorm, but let's understand it!

class LayerNorm(nn.Module):
    """
    Layer Normalization.
    Normalizes across the embedding dimension.
    
    Parameters:
    - embedding_dim: Dimension to normalize
    - eps: Small value to prevent division by zero
    """
    def __init__(self, embedding_dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        # Learnable scale and shift parameters
        self.gamma = nn.Parameter(torch.ones(embedding_dim))
        self.beta = nn.Parameter(torch.zeros(embedding_dim))
    
    def forward(self, x):
        """
        x: input, shape (batch_size, seq_len, embedding_dim)
        """
        # Calculate mean and std across embedding dimension
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True)
        
        # Normalize
        normalized = (x - mean) / (std + self.eps)
        
        # Scale and shift
        return self.gamma * normalized + self.beta

# Example:
layer_norm = LayerNorm(embedding_dim)

# Let's see the effect
sample = torch.randn(1, 5, embedding_dim) * 100  # Random values scaled by 100
normalized = layer_norm(sample)

print(f"Before normalization - Mean: {sample.mean():.2f}, Std: {sample.std():.2f}")
print(f"After normalization - Mean: {normalized.mean():.2f}, Std: {normalized.std():.2f}")
print("\nNotice how the values are now centered around 0 with std around 1!")

---
## Step 8: Building a Complete Decoder Block

Now we combine everything into one decoder block!

**Architecture of one block:**
```
Input
  ↓
Layer Norm
  ↓
Multi-Head Attention
  ↓
Add & Norm (residual connection)
  ↓
Layer Norm
  ↓
Feed-Forward Network
  ↓
Add & Norm (residual connection)
  ↓
Output
```

**Residual Connections (Add & Norm):**
- Instead of `output = layer(input)`
- We do `output = input + layer(input)`
- This helps gradients flow during training
- Think of it as allowing the model to keep the original information while adding new insights

In [ ]:
class DecoderBlock(nn.Module):
    """
    A single transformer decoder block.
    
    Parameters:
    - embedding_dim: Dimension of embeddings
    - num_heads: Number of attention heads
    - ff_dim: Feed-forward hidden dimension
    - dropout: Dropout probability
    """
    def __init__(self, embedding_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        
        # Multi-head attention
        self.attention = MultiHeadAttention(embedding_dim, num_heads, dropout)
        
        # Feed-forward network
        self.feed_forward = FeedForward(embedding_dim, ff_dim, dropout)
        
        # Layer normalization (we use pre-norm architecture)
        self.norm1 = nn.LayerNorm(embedding_dim)
        self.norm2 = nn.LayerNorm(embedding_dim)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        """
        x: input, shape (batch_size, seq_len, embedding_dim)
        mask: attention mask
        """
        # Step 1: Multi-head attention with residual connection
        # Pre-norm: normalize first, then apply attention
        norm_x = self.norm1(x)
        attention_output = self.attention(norm_x, mask)
        x = x + self.dropout(attention_output)  # Residual connection
        
        # Step 2: Feed-forward with residual connection
        norm_x = self.norm2(x)
        ff_output = self.feed_forward(norm_x)
        x = x + self.dropout(ff_output)  # Residual connection
        
        return x

# Example:
decoder_block = DecoderBlock(embedding_dim, num_heads, ff_dim)

block_output = decoder_block(embeddings_with_pos[:, :5, :], mask=causal_mask)
print(f"Decoder block output shape: {block_output.shape}")

# Count parameters
params = sum(p.numel() for p in decoder_block.parameters())
print(f"Parameters in one decoder block: {params:,}")

---
## Step 9: Complete Decoder-Only Transformer Model

**Final Architecture:**
```
Input Text → Tokenization → Token IDs
  ↓
Token Embedding + Positional Embedding
  ↓
Decoder Block 1
  ↓
Decoder Block 2
  ↓
...
  ↓
Decoder Block N
  ↓
Layer Norm
  ↓
Linear Layer (projects to vocabulary size)
  ↓
Output Logits (scores for each token in vocabulary)
```

**For 100M parameters, we'll configure:**
- Embedding dimension: 768
- Number of layers: 12
- Number of heads: 12
- Feed-forward dimension: 3072 (4 * 768)

In [ ]:
class DecoderTransformer(nn.Module):
    """
    Complete Decoder-Only Transformer Model (like GPT).
    
    Parameters:
    - vocab_size: Size of vocabulary
    - embedding_dim: Dimension of embeddings
    - num_layers: Number of decoder blocks
    - num_heads: Number of attention heads
    - ff_dim: Feed-forward hidden dimension
    - max_seq_len: Maximum sequence length
    - dropout: Dropout probability
    """
    def __init__(self, vocab_size, embedding_dim, num_layers, num_heads, 
                 ff_dim, max_seq_len, dropout=0.1):
        super().__init__()
        
        self.embedding_dim = embedding_dim
        self.max_seq_len = max_seq_len
        
        # Token and positional embeddings
        self.token_embedding = TokenEmbedding(vocab_size, embedding_dim)
        self.pos_embedding = PositionalEmbedding(max_seq_len, embedding_dim)
        
        # Stack of decoder blocks
        self.decoder_blocks = nn.ModuleList([
            DecoderBlock(embedding_dim, num_heads, ff_dim, dropout)
            for _ in range(num_layers)
        ])
        
        # Final layer norm
        self.final_norm = nn.LayerNorm(embedding_dim)
        
        # Output projection to vocabulary
        self.output_projection = nn.Linear(embedding_dim, vocab_size)
        
        # Dropout
        self.dropout = nn.Dropout(dropout)
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        """Initialize weights for better training."""
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def create_causal_mask(self, seq_len, device):
        """Create causal mask for decoder (can't attend to future positions)."""
        mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
        return mask.unsqueeze(0)  # Add batch dimension
    
    def forward(self, x):
        """
        x: input token IDs, shape (batch_size, seq_len)
        returns: logits, shape (batch_size, seq_len, vocab_size)
        """
        batch_size, seq_len = x.shape
        
        # Step 1: Get embeddings
        token_emb = self.token_embedding(x)  # (batch_size, seq_len, embedding_dim)
        x = self.pos_embedding(token_emb)     # Add positional embeddings
        x = self.dropout(x)
        
        # Step 2: Create causal mask
        mask = self.create_causal_mask(seq_len, x.device)
        
        # Step 3: Pass through decoder blocks
        for decoder_block in self.decoder_blocks:
            x = decoder_block(x, mask)
        
        # Step 4: Final layer norm
        x = self.final_norm(x)
        
        # Step 5: Project to vocabulary
        logits = self.output_projection(x)  # (batch_size, seq_len, vocab_size)
        
        return logits

# Create the model with ~100M parameters
model_config = {
    'vocab_size': 50,  # We'll use a larger vocab with real data
    'embedding_dim': 768,
    'num_layers': 12,
    'num_heads': 12,
    'ff_dim': 3072,  # 4 * embedding_dim
    'max_seq_len': 512,
    'dropout': 0.1
}

model = DecoderTransformer(**model_config)

# Count total parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"\nModel size: ~{total_params / 1e6:.1f}M parameters")

# Test the model
test_input = torch.randint(0, 50, (2, 10))  # Batch of 2, sequence length 10
output = model(test_input)
print(f"\nInput shape: {test_input.shape}")
print(f"Output shape: {output.shape}")
print(f"Output shape means: (batch_size={output.shape[0]}, seq_len={output.shape[1]}, vocab_size={output.shape[2]})")

---
## Step 10: Preparing Training Data

**How does the model learn?**

The model learns by predicting the next token:

**Example:**
- Input: "The cat sat on the"
- Target: "cat sat on the mat"

At each position, the model predicts the next token:
- Position 0: Given "The" → Predict "cat"
- Position 1: Given "The cat" → Predict "sat"
- Position 2: Given "The cat sat" → Predict "on"
- And so on...

**This is called "teacher forcing"** - we give the model the correct previous tokens during training.

In [ ]:
class TextDataset(Dataset):
    """
    Dataset for language modeling.
    Creates input-target pairs for next token prediction.
    
    Parameters:
    - text: Input text string
    - tokenizer: Tokenizer to convert text to IDs
    - seq_len: Length of each sequence
    """
    def __init__(self, text, tokenizer, seq_len):
        self.tokenizer = tokenizer
        self.seq_len = seq_len
        
        # Encode the entire text
        self.token_ids = tokenizer.encode(text)
        
    def __len__(self):
        # Number of sequences we can create
        return len(self.token_ids) - self.seq_len
    
    def __getitem__(self, idx):
        """
        Returns input and target sequences.
        Target is the input shifted by one position.
        """
        # Get a sequence of seq_len + 1 tokens
        chunk = self.token_ids[idx:idx + self.seq_len + 1]
        
        # Input: first seq_len tokens
        x = torch.tensor(chunk[:-1], dtype=torch.long)
        
        # Target: last seq_len tokens (shifted by 1)
        y = torch.tensor(chunk[1:], dtype=torch.long)
        
        return x, y

# Example with a simple text
training_text = """
Once upon a time, in a land far away, there lived a wise old wizard.
The wizard had a magical staff that could create wonderful things.
One day, a young apprentice came to learn magic from the wizard.
The wizard taught the apprentice many spells and tricks.
Together, they went on many adventures and helped people in need.
The apprentice learned that the greatest magic comes from kindness.
"""

# Create tokenizer
simple_tokenizer = CharacterTokenizer(training_text)
print(f"Vocabulary size: {simple_tokenizer.vocab_size}")

# Create dataset
seq_len = 64  # Length of each training sequence
dataset = TextDataset(training_text, simple_tokenizer, seq_len)
print(f"\nDataset size: {len(dataset)} sequences")

# Example of one training sample
x_sample, y_sample = dataset[0]
print(f"\nExample training pair:")
print(f"Input:  '{simple_tokenizer.decode(x_sample.tolist())}'")
print(f"Target: '{simple_tokenizer.decode(y_sample.tolist())}'")
print(f"\nNotice: Target is input shifted by one character!")

---
## Step 11: Training Loop

**How training works:**

1. **Forward Pass**: Feed input through model to get predictions
2. **Calculate Loss**: Compare predictions with actual next tokens
3. **Backward Pass**: Calculate gradients (how to adjust weights)
4. **Update Weights**: Adjust model parameters to reduce loss
5. **Repeat**: Do this many times!

**Loss Function:**
- We use Cross-Entropy Loss
- It measures how different predictions are from actual targets
- Lower loss = better predictions

**Optimizer:**
- We use AdamW (Adam with weight decay)
- It's like a smart way to adjust weights
- Learning rate controls how big each update is

In [ ]:
def train_model(model, dataset, epochs, batch_size, learning_rate, device):
    """
    Train the transformer model.
    
    Parameters:
    - model: The transformer model
    - dataset: Training dataset
    - epochs: Number of training epochs
    - batch_size: Batch size
    - learning_rate: Learning rate
    - device: Device to train on (CPU or GPU)
    """
    model = model.to(device)
    model.train()
    
    # Create data loader
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    # Loss function
    criterion = nn.CrossEntropyLoss()
    
    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    
    # Training loop
    for epoch in range(epochs):
        total_loss = 0
        num_batches = 0
        
        for batch_idx, (x, y) in enumerate(dataloader):
            # Move data to device
            x = x.to(device)
            y = y.to(device)
            
            # Forward pass
            logits = model(x)  # Shape: (batch_size, seq_len, vocab_size)
            
            # Reshape for loss calculation
            # CrossEntropyLoss expects (batch_size * seq_len, vocab_size)
            logits = logits.reshape(-1, logits.size(-1))
            y = y.reshape(-1)
            
            # Calculate loss
            loss = criterion(logits, y)
            
            # Backward pass
            optimizer.zero_grad()  # Clear previous gradients
            loss.backward()        # Calculate gradients
            
            # Gradient clipping (prevents exploding gradients)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # Update weights
            optimizer.step()
            
            total_loss += loss.item()
            num_batches += 1
            
            # Print progress
            if (batch_idx + 1) % 10 == 0:
                print(f"Epoch [{epoch+1}/{epochs}], Batch [{batch_idx+1}/{len(dataloader)}], "
                      f"Loss: {loss.item():.4f}")
        
        # Print epoch summary
        avg_loss = total_loss / num_batches
        print(f"\nEpoch {epoch+1} Complete - Average Loss: {avg_loss:.4f}\n")
    
    return model

print("Training function defined!")
print("\nWe'll train a smaller model for demonstration purposes.")
print("For a full 100M model, you'd need more data and compute time.")

---
## Step 12: Text Generation

**How to generate text:**

1. Start with a prompt (e.g., "Once upon a")
2. Feed it through the model
3. Get predictions for next token
4. Sample a token from predictions
5. Add it to the sequence
6. Repeat!

**Sampling Strategies:**

1. **Greedy**: Always pick the most likely token (boring, repetitive)
2. **Temperature Sampling**: 
   - Temperature > 1: More random, creative
   - Temperature < 1: More focused, conservative
   - Temperature = 1: Normal probabilities
3. **Top-k Sampling**: Only consider top k most likely tokens
4. **Top-p (Nucleus) Sampling**: Consider smallest set of tokens with cumulative probability > p

In [ ]:
def generate_text(model, tokenizer, prompt, max_new_tokens, temperature=1.0, 
                  top_k=None, device='cpu'):
    """
    Generate text using the trained model.
    
    Parameters:
    - model: Trained transformer model
    - tokenizer: Tokenizer for encoding/decoding
    - prompt: Starting text
    - max_new_tokens: Number of tokens to generate
    - temperature: Sampling temperature (higher = more random)
    - top_k: If set, only sample from top k tokens
    - device: Device to run on
    """
    model.eval()  # Set to evaluation mode
    model = model.to(device)
    
    # Encode the prompt
    token_ids = tokenizer.encode(prompt)
    x = torch.tensor([token_ids], dtype=torch.long).to(device)
    
    # Generate tokens one by one
    with torch.no_grad():  # No need to track gradients
        for _ in range(max_new_tokens):
            # Get predictions
            logits = model(x)  # (1, seq_len, vocab_size)
            
            # Focus on the last position
            logits = logits[:, -1, :]  # (1, vocab_size)
            
            # Apply temperature
            logits = logits / temperature
            
            # Apply top-k filtering if specified
            if top_k is not None:
                top_k_logits, top_k_indices = torch.topk(logits, top_k)
                logits = torch.full_like(logits, float('-inf'))
                logits.scatter_(1, top_k_indices, top_k_logits)
            
            # Convert to probabilities
            probs = F.softmax(logits, dim=-1)
            
            # Sample from the distribution
            next_token = torch.multinomial(probs, num_samples=1)
            
            # Append to sequence
            x = torch.cat([x, next_token], dim=1)
            
            # Stop if we exceed max sequence length
            if x.size(1) >= model.max_seq_len:
                break
    
    # Decode and return
    generated_ids = x[0].tolist()
    generated_text = tokenizer.decode(generated_ids)
    
    return generated_text

print("Text generation function defined!")
print("\nWe can use different temperature values:")
print("- temperature=0.5: More focused, less random")
print("- temperature=1.0: Balanced")
print("- temperature=1.5: More creative, more random")

---
## Step 13: Train a Small Demo Model

Let's train a small version to see it work! For a full 100M parameter model, you'd need:
- Much more training data (millions/billions of tokens)
- Powerful GPU(s)
- Hours/days of training time

We'll train a tiny model for demonstration:

In [ ]:
# Create a smaller model for quick training
small_model_config = {
    'vocab_size': simple_tokenizer.vocab_size,
    'embedding_dim': 128,   # Smaller
    'num_layers': 4,        # Fewer layers
    'num_heads': 4,         # Fewer heads
    'ff_dim': 512,          # Smaller FFN
    'max_seq_len': 128,
    'dropout': 0.1
}

small_model = DecoderTransformer(**small_model_config)

# Count parameters
small_params = sum(p.numel() for p in small_model.parameters())
print(f"Small model parameters: {small_params:,}")
print(f"Small model size: ~{small_params / 1e6:.2f}M parameters")

# Prepare for training
print("\n" + "="*50)
print("Starting training...")
print("="*50 + "\n")

# Train the model
small_model = train_model(
    model=small_model,
    dataset=dataset,
    epochs=50,              # More epochs for small data
    batch_size=16,
    learning_rate=3e-4,
    device=device
)

print("\n" + "="*50)
print("Training complete!")
print("="*50)

---
## Step 14: Generate Text with Our Trained Model

Now let's see what our model learned!

In [ ]:
# Test text generation with different prompts and temperatures

prompts = [
    "Once upon a time",
    "The wizard",
    "The apprentice learned"
]

temperatures = [0.5, 1.0, 1.5]

print("\n" + "="*70)
print("TEXT GENERATION EXAMPLES")
print("="*70 + "\n")

for prompt in prompts:
    print(f"\nPrompt: '{prompt}'\n")
    print("-" * 70)
    
    for temp in temperatures:
        generated = generate_text(
            model=small_model,
            tokenizer=simple_tokenizer,
            prompt=prompt,
            max_new_tokens=100,
            temperature=temp,
            top_k=20,
            device=device
        )
        
        print(f"\nTemperature {temp}:")
        print(generated)
        print()
    
    print("=" * 70)

---
## Step 15: Understanding Model Architecture - 100M Parameters

Let's analyze how to configure a proper 100M parameter model:

In [ ]:
def count_parameters(config):
    """
    Estimate parameter count for a given configuration.
    """
    vocab_size = config['vocab_size']
    d_model = config['embedding_dim']
    n_layers = config['num_layers']
    n_heads = config['num_heads']
    d_ff = config['ff_dim']
    
    # Token embedding
    token_emb_params = vocab_size * d_model
    
    # Positional embedding
    pos_emb_params = config['max_seq_len'] * d_model
    
    # One decoder block parameters
    # Multi-head attention: Q, K, V projections + output projection
    mha_params = 4 * (d_model * d_model)
    
    # Feed-forward network: 2 linear layers
    ffn_params = (d_model * d_ff) + (d_ff * d_model)
    
    # Layer norms (2 per block): scale and bias
    ln_params = 2 * 2 * d_model
    
    # Total per block
    block_params = mha_params + ffn_params + ln_params
    
    # All blocks
    all_blocks_params = n_layers * block_params
    
    # Final layer norm
    final_ln_params = 2 * d_model
    
    # Output projection (often shares weights with token embedding, but let's count it)
    output_params = d_model * vocab_size
    
    # Total
    total = (token_emb_params + pos_emb_params + all_blocks_params + 
             final_ln_params + output_params)
    
    return {
        'token_embedding': token_emb_params,
        'pos_embedding': pos_emb_params,
        'all_blocks': all_blocks_params,
        'per_block': block_params,
        'final_ln': final_ln_params,
        'output_projection': output_params,
        'total': total
    }

# Configuration for ~100M parameters
config_100m = {
    'vocab_size': 50000,      # Typical for a real tokenizer (BPE)
    'embedding_dim': 768,     # Standard for mid-size models
    'num_layers': 12,         # 12 transformer blocks
    'num_heads': 12,          # 12 attention heads
    'ff_dim': 3072,           # 4 * embedding_dim
    'max_seq_len': 1024,      # Support 1024 tokens
    'dropout': 0.1
}

params_breakdown = count_parameters(config_100m)

print("\n" + "="*70)
print("100M PARAMETER MODEL CONFIGURATION")
print("="*70 + "\n")

print("Configuration:")
for key, value in config_100m.items():
    print(f"  {key:20s}: {value:,}")

print("\n" + "-"*70)
print("Parameter Breakdown:")
print("-"*70 + "\n")

print(f"Token Embedding:      {params_breakdown['token_embedding']:>15,} parameters")
print(f"Positional Embedding: {params_breakdown['pos_embedding']:>15,} parameters")
print(f"All Decoder Blocks:   {params_breakdown['all_blocks']:>15,} parameters")
print(f"  (per block:         {params_breakdown['per_block']:>15,} parameters)")
print(f"Final Layer Norm:     {params_breakdown['final_ln']:>15,} parameters")
print(f"Output Projection:    {params_breakdown['output_projection']:>15,} parameters")
print("-"*70)
print(f"TOTAL:                {params_breakdown['total']:>15,} parameters")
print(f"                      {params_breakdown['total']/1e6:>15.1f}M parameters")
print("="*70)

print("\n💡 Note: For actual training, you'd need:")
print("   - Large dataset (GB to TB of text)")
print("   - Powerful GPU (A100, H100, etc.)")
print("   - Days to weeks of training time")
print("   - Careful hyperparameter tuning")
print("   - Learning rate scheduling")
print("   - Proper tokenizer (BPE, WordPiece, etc.)")

---
## Summary: What We Built

Congratulations! You now understand every component of a decoder-only transformer:

### Components:
1. ✅ **Tokenization**: Converting text to numbers
2. ✅ **Token Embeddings**: Converting token IDs to dense vectors
3. ✅ **Positional Embeddings**: Adding position information
4. ✅ **Self-Attention**: Letting positions attend to each other
5. ✅ **Multi-Head Attention**: Multiple attention perspectives
6. ✅ **Feed-Forward Network**: Processing and transforming information
7. ✅ **Layer Normalization**: Stabilizing training
8. ✅ **Decoder Block**: Combining all components
9. ✅ **Full Transformer**: Stacking blocks into a complete model
10. ✅ **Training**: Teaching the model to predict next tokens
11. ✅ **Generation**: Creating new text

### Key Insights:

**Attention Mechanism**: The core innovation that allows the model to focus on relevant information.

**Causal Masking**: Essential for decoder models - prevents cheating by looking at future tokens.

**Residual Connections**: Help gradients flow during training, enable deeper models.

**Scaling**: More parameters = more capacity, but needs more data and compute.

### Next Steps:

1. **Better Tokenization**: Use BPE (Byte-Pair Encoding) or SentencePiece
2. **More Data**: Train on large text corpora (books, websites, etc.)
3. **Optimization**: Learn about learning rate scheduling, gradient accumulation
4. **Advanced Techniques**: Flash attention, mixed precision training
5. **Evaluation**: Perplexity, benchmark tasks

### Famous Decoder-Only Models:
- **GPT-2**: 1.5B parameters
- **GPT-3**: 175B parameters
- **GPT-4**: Estimated >1T parameters
- **LLaMA**: 7B to 65B parameters

You now have the foundation to understand how these models work! 🎉

---
## Bonus: Interactive Exploration

Use the cells below to experiment!

In [ ]:
# Try different prompts here!
your_prompt = "The wizard said"

result = generate_text(
    model=small_model,
    tokenizer=simple_tokenizer,
    prompt=your_prompt,
    max_new_tokens=150,
    temperature=1.0,
    top_k=30,
    device=device
)

print(result)

In [ ]:
# Visualize attention patterns (advanced)
import matplotlib.pyplot as plt
import seaborn as sns

def visualize_attention(model, text, tokenizer, device):
    """Visualize attention weights for a given text."""
    model.eval()
    
    # Encode text
    tokens = tokenizer.encode(text[:50])  # First 50 chars
    x = torch.tensor([tokens], dtype=torch.long).to(device)
    
    # We'd need to modify the model to return attention weights
    # This is a placeholder for the concept
    print("To visualize attention, we'd modify the model to return attention weights.")
    print("This would show which positions the model focuses on for each token.")
    
# visualize_attention(small_model, "The wizard", simple_tokenizer, device)